# Wildlife identification from a photo

Use Claude's vision capability via the Messages API to identify wildlife in a photograph and
return a structured analysis: subject detection, species identification, plausible look-alikes,
habitat cues, and a 1-4 confidence rating.

**Requirements:** an `ANTHROPIC_API_KEY` in your environment (or a `.env` file at the repo
root). See the repository README for setup.

In [ ]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def url_image_block(url):
    """Wrap a public image URL as an Anthropic image content block.
    Claude fetches the bytes itself, so nothing needs to live on disk."""
    return {"type": "image", "source": {"type": "url", "url": url}}


def image_block(path, media_type="image/jpeg"):
    """Wrap a local image file as a base64 image content block.
    Use this for your own photos that are not reachable by URL."""
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")
    return {
        "type": "image",
        "source": {"type": "base64", "media_type": media_type, "data": data},
    }

## The image source

The Messages API can analyze an image supplied either as base64-encoded bytes from a local
file or fetched directly from a public URL. This notebook uses a URL so it stays
self-contained with no local file to manage.

The photo below is a red fox from the U.S. National Park Service (Cedar Breaks National
Monument). NPS photographs are works of the federal government and are in the public domain,
so they are safe to use in examples and teaching material.

Source: https://www.nps.gov/cebr/learn/nature/red-fox.htm

To run the same analysis on your own photo, swap `url_image_block(...)` for
`image_block("your_photo.jpg")` further down.

In [ ]:
# Wildlife identification prompt
prompt = """
Analyze the attached wildlife photo with these specific steps. Identify only what the image
actually supports, and say so plainly when a feature is obscured or ambiguous.

1. Subject detection: Establish what is in the frame:
   - How many animals are present and where they sit in the frame
   - How much of each animal is visible (full body, head only, partially occluded)
   - Overall image quality factors that affect identification (lighting, focus, distance)

2. Identification: Name the animal as precisely as the image allows:
   - The most likely common name, and the species (binomial name) if you are confident enough
   - The specific visual features that drive the identification (coat color and pattern, ear
     shape, snout, tail, leg markings, relative size, body proportions)

3. Alternatives and confounders: Guard against overconfidence:
   - List the most plausible look-alike species
   - For each, name the feature in the photo that argues for or against it

4. Habitat and context cues: Read the surroundings:
   - Describe the environment, substrate, vegetation, and the animal's posture or behavior
   - Note what these cues suggest about the setting or the animal's identity

5. Identification Confidence Rating: Assign a rating from 1-4:
   - Rating 1 (Tentative): Only a broad category is supportable (for example, "a canid");
     key diagnostic features are obscured.
   - Rating 2 (Plausible): A likely species, but strong look-alikes cannot be ruled out.
   - Rating 3 (Confident): Species identification is well supported by multiple distinguishing
     features.
   - Rating 4 (Definitive): Unambiguous; diagnostic features are clearly visible and no
     realistic alternative remains.

For each item above (1-5), write one sentence summarizing your findings, with your final
response being the numeric Identification Confidence Rating (1-4) with a brief justification.
"""

In [ ]:
# Feed the image into Claude
image_url = "https://www.nps.gov/cebr/learn/nature/images/RedFox_4.jpg"

messages = []
add_user_message(
    messages,
    [
        url_image_block(image_url),
        {"type": "text", "text": prompt},
    ],
)

# To use a local photo instead, comment out the block above and use:
# add_user_message(
#     messages,
#     [image_block("your_photo.jpg"), {"type": "text", "text": prompt}],
# )

response = chat(messages)
print(text_from_message(response))